# HOSPITAL OPERATIONS & READMISSION ANALYSIS

## 1. Import Libraries and Set Display

In [60]:
# Libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Display
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pio.templates.default = "plotly_white"

## 2. Load and Clean Data

In [61]:
# Load raw data
df = pd.read_csv("hospital_stays.csv")

# Set datatype
df["admission_date"] = pd.to_datetime(df["admission_date"])
df["discharge_date"] = pd.to_datetime(df["discharge_date"])

# Make new columns
age_bins = [0, 1, 5, 12, 18, 40, 65, 120]
age_labels = ["Infant", "Toddler", "Child",
              "Teen", "Adult", "Middle-age", "Senior"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels, right=True)

df["charge_per_day"] = (df["total_charges"] / df["length_of_stay"]).round(0)

df['insurance_type'] = df['insurance_type'].replace('Asuransi Lain', 'Others')

print(f"Dataset loaded and cleaned: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

Dataset loaded and cleaned: 10,500 rows, 17 columns


,patient_id,age,gender,blood_type,bmi,smoker,admission_date,discharge_date,length_of_stay,diagnosis_code,diagnosis,department,total_charges,insurance_type,readmitted_30d,age_group,charge_per_day
0,P00001,4,Female,A-,33.10,No,2023-09-08,2023-09-10,2,I25,Coronary Artery Disease,Cardiology,2907473,BPJS Kesehatan,No,Toddler,"1,453,736.00"
1,P00002,4,Male,O-,33.10,Yes,2024-10-27,2024-11-06,10,I21,Myocardial Infarction,Cardiology,15031148,BPJS Kesehatan,Yes,Toddler,"1,503,115.00"
2,P00003,44,Female,O+,18.80,Yes,2023-06-09,2023-06-15,6,I63,Stroke Ischemic,Neurology,11897251,Corporate,Yes,Middle-age,"1,982,875.00"
3,P00004,69,Male,AB-,32.70,Yes,2024-01-23,2024-01-27,4,K80,Cholelithiasis,Gastroenterology,5307419,BPJS Kesehatan,No,Senior,"1,326,855.00"
4,P00005,11,Male,AB-,20.70,No,2023-04-14,2023-04-18,4,M17,Osteoarthritis,Orthopedics,5875435,Corporate,No,Child,"1,468,859.00"


# 3. Analysis


## 3.1 Executive KPIs


In [62]:
total_admissions = len(df)
total_revenue = df["total_charges"].sum()
avg_los = df["length_of_stay"].mean() # LOS = Length of stay
median_los = df["length_of_stay"].median()
avg_charge_per_stay = df["total_charges"].mean()
readmission_rate = (df["readmitted_30d"] == "Yes").mean()

# Benchmarks
readmission_benchmark = 0.15  # International hospital benchmark
readmission_vs_benchmark = readmission_rate - readmission_benchmark

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Admissions", "Total Revenue (IDR)", "Avg Length of Stay (days)",
        "Median Length of Stay (days)", "Avg Charge per Stay (IDR)",
        "30-Day Readmission Rate", "vs. 15% Benchmark"
    ],
    "Value": [
        f"{total_admissions:,}", f"{total_revenue:,.0f}", f"{avg_los:.1f}",
        f"{median_los:.1f}", f"{avg_charge_per_stay:,.0f}",
        f"{readmission_rate:.1%}", f"{readmission_vs_benchmark:+.1%}"
    ]
})

kpi_summary

,Metric,Value
0,Total Admissions,"10,500"
1,Total Revenue (IDR),"103,677,124,132"
2,Avg Length of Stay (days),6.9
3,Median Length of Stay (days),7.0
4,Avg Charge per Stay (IDR),"9,874,012"
5,30-Day Readmission Rate,19.1%
6,vs. 15% Benchmark,+4.1%


## 3.2 Monthly Trend

In [63]:
monthly_trend = (
    df.set_index("admission_date")
      .resample("ME")
      .agg(
          admissions=("patient_id", "count"),
          revenue=("total_charges", "sum"),
          avg_los=("length_of_stay", "mean")
      )
      .reset_index()
)

monthly_trend.head()

,admission_date,admissions,revenue,avg_los
0,2023-01-31,463,4543261177,6.84
1,2023-02-28,358,3567524848,7.05
2,2023-03-31,450,4482387430,6.98
3,2023-04-30,484,4764215442,6.85
4,2023-05-31,458,4342359864,6.71


## 3.3 Insurance Type

In [64]:
insurance_type_summary = (
    df.groupby("insurance_type")
      .agg(
          stays=("patient_id", "count"),
          revenue=("total_charges", "sum")
      )
      .assign(
          pct_of_stays=lambda x: x["stays"] / x["stays"].sum(),
          pct_of_revenue=lambda x: x["revenue"] / x["revenue"].sum()
      )
      .sort_values("stays", ascending=False)
      .reset_index()
)

insurance_type_summary

,insurance_type,stays,revenue,pct_of_stays,pct_of_revenue
0,BPJS Kesehatan,4243,41644265344,0.40,0.40
1,Private,2023,20175570142,0.19,0.19
2,Self Pay,1617,16341440822,0.15,0.16
3,Corporate,1583,15778918228,0.15,0.15
4,Others,1034,9736929596,0.10,0.09


## 3.4 Monthly Admissions Trend

In [65]:
fig_trend = px.line(
    monthly_trend,
    x="admission_date",
    y="admissions",
    title="Monthly Admissions Trend (2023–2025)",
    labels={"admission_date": "Month", "admissions": "Admissions"},
    markers=True
)

fig_trend.update_traces(line_color=PRIMARY_COLOR, line_width=2.5)
fig_trend.update_layout(
    hovermode="x unified",
    yaxis_title="Number of Admissions",
    xaxis_title=None
)

fig_trend.show()

## 3.5 Admissions by Department

In [66]:
dept_counts = (
    df["department"]
      .value_counts()
      .reset_index()
      .rename(columns={"count": "admissions"})
      .sort_values("admissions", ascending=True)  # ascending for horizontal bar = top at top
)

fig_dept = px.bar(
    dept_counts,
    x="admissions",
    y="department",
    orientation="h",
    title="Admissions by Department",
    color="department",
    color_discrete_map=DEPARTMENT_COLORS,
    text="admissions"
)

fig_dept.update_layout(showlegend=False, xaxis_title="Admissions", yaxis_title=None)
fig_dept.update_traces(textposition="outside")

fig_dept.show()

## 3.6 Insurance Type by Volume

In [67]:
fig_insurance_type = px.pie(
    insurance_type_summary,
    values="stays",
    names="insurance_type",
    title="Insurance Type by Volume",
    hole=0.5
)

fig_insurance_type.update_traces(
    textinfo="label+percent",
    textposition="outside"
)
fig_insurance_type.update_layout(showlegend=False)

fig_insurance_type.show()

## 3.7 KPI Cards Preview

In [68]:
print("EXECUTIVE SUMMARY")
print("=" * 50)
print(f"Total Admissions:        {total_admissions:,}")
print(f"Total Revenue:           IDR {total_revenue:,.0f}")
print(f"Avg Length of Stay:      {avg_los:.1f} days (median: {median_los:.1f})")
print(f"30-Day Readmission Rate: {readmission_rate:.1%} "
      f"({readmission_vs_benchmark:+.1%} vs. 15% benchmark)")

EXECUTIVE SUMMARY
Total Admissions:        10,500
Total Revenue:           IDR 103,677,124,132
Avg Length of Stay:      6.9 days (median: 7.0)
30-Day Readmission Rate: 19.1% (+4.1% vs. 15% benchmark)


## 3.8 Length of Stay by Department

In [69]:
fig_los_dept = px.box(
    df,
    x="department",
    y="length_of_stay",
    title="Length of Stay Distribution by Department",
    color="department",
    color_discrete_map=DEPARTMENT_COLORS,
    points=False
)

fig_los_dept.update_layout(
    showlegend=False,
    xaxis_title=None,
    yaxis_title="Length of Stay (days)"
)

fig_los_dept.show()

## 3.9 Admissions Heatmap

In [70]:
df["month"] = df["admission_date"].dt.to_period("M").astype(str)

heatmap_data = (
    df.groupby(["month", "department"])
      .size()
      .reset_index(name="admissions")
      .pivot(index="department", columns="month", values="admissions")
      .fillna(0)
)

fig_heatmap = px.imshow(
    heatmap_data,
    title="Admissions Heatmap — Department × Month",
    labels=dict(x="Month", y="Department", color="Admissions"),
    color_continuous_scale="Teal",
    aspect="auto"
)

fig_heatmap.update_layout(xaxis_tickangle=-45)

fig_heatmap.show()

## 3.10 Total Bed-days by Department

In [71]:
bed_days = (
    df.groupby("department")["length_of_stay"]
      .sum()
      .reset_index(name="total_bed_days")
      .sort_values("total_bed_days", ascending=True)
)

fig_beddays = px.bar(
    bed_days,
    x="total_bed_days",
    y="department",
    orientation="h",
    title="Total Bed-Days Consumed by Department",
    color="department",
    color_discrete_map=DEPARTMENT_COLORS,
    text="total_bed_days"
)

fig_beddays.update_layout(showlegend=False, xaxis_title="Total Bed-Days", yaxis_title=None)
fig_beddays.update_traces(textposition="outside")

fig_beddays.show()

## 3.11 Admissions by Day

In [72]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_counts = (
    df["admission_date"].dt.day_name()
      .value_counts()
      .reindex(weekday_order)
      .reset_index()
)
weekday_counts.columns = ["weekday", "admissions"]

fig_weekday = px.bar(
    weekday_counts,
    x="weekday",
    y="admissions",
    title="Admissions by Day of Week",
    color_discrete_sequence=[PRIMARY_COLOR]
)

fig_weekday.update_layout(xaxis_title=None, yaxis_title="Admissions")

fig_weekday.show()

## 3.12 Admission Volume by Diagnosis

In [73]:
diagnosis_counts = (
    df["diagnosis"]
      .value_counts()
      .reset_index()
      .rename(columns={"count": "admissions"})
      .sort_values("admissions", ascending=True)
)

fig_diag_volume = px.bar(
    diagnosis_counts,
    x="admissions",
    y="diagnosis",
    orientation="h",
    title="Admission Volume by Diagnosis",
    color_discrete_sequence=[PRIMARY_COLOR]
)

fig_diag_volume.update_layout(yaxis_title=None, xaxis_title="Admissions")

fig_diag_volume.show()

## 3.13 Average Length of Stay by Diagnosis

In [74]:
los_by_diagnosis = (
    df.groupby("diagnosis")["length_of_stay"]
      .mean()
      .round(1)
      .reset_index()
      .sort_values("length_of_stay", ascending=True)
)

fig_los_diag = px.bar(
    los_by_diagnosis,
    x="length_of_stay",
    y="diagnosis",
    orientation="h",
    title="Average Length of Stay by Diagnosis",
    color_discrete_sequence=[PRIMARY_COLOR]
)

fig_los_diag.update_layout(yaxis_title=None, xaxis_title="Avg Length of Stay (days)")

fig_los_diag.show()

## 3.14 Monthly Revenue Trend

In [75]:
fig_revenue_trend = px.line(
    monthly_trend,
    x="admission_date",
    y="revenue",
    title="Monthly Revenue Trend (2023–2025)",
    labels={"admission_date": "Month", "revenue": "Revenue (IDR)"},
    markers=True
)

fig_revenue_trend.update_traces(line_color=PRIMARY_COLOR, line_width=2.5)
fig_revenue_trend.update_layout(
    hovermode="x unified",
    yaxis_title="Total Revenue (IDR)",
    xaxis_title=None
)

fig_revenue_trend.show()

## 3.15 Revenue by Department

In [76]:
dept_revenue = (
    df.groupby("department")["total_charges"]
      .agg(total_revenue="sum", avg_revenue="mean")
      .reset_index()
      .sort_values("total_revenue", ascending=True)
)
fig_rev_total = px.bar(
    dept_revenue, x="total_revenue", y="department", orientation="h",
    title="Total Revenue by Department",
    color="department", color_discrete_map=DEPARTMENT_COLORS, text="total_revenue"
)
fig_rev_total.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig_rev_total.update_layout(showlegend=False, xaxis_title="Total Revenue (IDR)", yaxis_title=None)
fig_rev_total.show()

fig_rev_avg = px.bar(
    dept_revenue.sort_values("avg_revenue", ascending=True),
    x="avg_revenue", y="department", orientation="h",
    title="Average Revenue per Stay by Department",
    color="department", color_discrete_map=DEPARTMENT_COLORS, text="avg_revenue"
)
fig_rev_avg.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig_rev_avg.update_layout(showlegend=False, xaxis_title="Avg Revenue per Stay (IDR)", yaxis_title=None)
fig_rev_avg.show()

## 3.16 Average Charge per Day by Department

In [77]:
charge_per_day_dept = (
    df.groupby("department")["charge_per_day"]
      .mean()
      .round(0)
      .reset_index()
      .sort_values("charge_per_day", ascending=True)
)

fig_charge_day = px.bar(
    charge_per_day_dept,
    x="charge_per_day", y="department", orientation="h",
    title="Average Charge per Day by Department",
    color_discrete_sequence=[PRIMARY_COLOR],
    text="charge_per_day"
)
fig_charge_day.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig_charge_day.update_layout(xaxis_title="Avg Charge per Day (IDR)", yaxis_title=None)

fig_charge_day.show()

## 3.17 Average Length of Stay by Insurance Type

In [78]:
insurance_stats = (
    df.groupby("insurance_type")
      .agg(avg_charges=("total_charges", "mean"), avg_los=("length_of_stay", "mean"))
      .round(1)
      .reset_index()
      .sort_values("avg_charges", ascending=False)
)

fig_ins_charges = px.bar(
    insurance_stats, x="insurance_type", y="avg_charges",
    title="Average Charges by Insurance Type",
    color_discrete_sequence=[PRIMARY_COLOR], text="avg_charges"
)
fig_ins_charges.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig_ins_charges.update_layout(xaxis_title=None, yaxis_title="Avg Charges (IDR)")
fig_ins_charges.show()

fig_ins_los = px.bar(
    insurance_stats.sort_values("avg_los", ascending=False),
    x="insurance_type", y="avg_los",
    title="Average Length of Stay by Insurance Type",
    color_discrete_sequence=["#5a9bb0"], text="avg_los"
)
fig_ins_los.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig_ins_los.update_layout(xaxis_title=None, yaxis_title="Avg LOS (days)")
fig_ins_los.show()

## 3.18 Revenue by Diagnosis

In [79]:
diagnosis_revenue = (
    df.groupby("diagnosis")["total_charges"]
      .sum()
      .reset_index(name="total_revenue")
      .sort_values("total_revenue", ascending=False)
)

fig_treemap = px.treemap(
    diagnosis_revenue,
    path=["diagnosis"],
    values="total_revenue",
    title="Revenue Share by Diagnosis",
    color="total_revenue",
    color_continuous_scale="Teal"
)

fig_treemap.update_traces(textinfo="label+percent root")

fig_treemap.show()

## 3.19 Readmission Rate by Department

In [80]:
readmit_by_dept = (
    df.groupby("department")["readmitted_30d"]
      .apply(lambda x: (x == "Yes").mean())
      .reset_index(name="readmission_rate")
      .sort_values("readmission_rate", ascending=True)
)

fig_readmit_dept = px.bar(
    readmit_by_dept,
    x="readmission_rate", y="department", orientation="h",
    title="30-Day Readmission Rate by Department",
    color_discrete_sequence=[ALERT_COLOR],
    text="readmission_rate"
)
fig_readmit_dept.update_traces(texttemplate="%{text:.1%}", textposition="outside")

fig_readmit_dept.add_vline(
    x=readmission_benchmark,
    line_dash="dash",
    line_color="gray",
    annotation_text="Benchmark (15%)",
    annotation_position="top"
)

fig_readmit_dept.update_layout(xaxis_title="Readmission Rate", yaxis_title=None, xaxis_tickformat=".0%")

fig_readmit_dept.show()

## 3.20 Readmission Rate Heatmap

In [81]:
readmit_heatmap_data = (
    df.groupby(["department", "diagnosis"])["readmitted_30d"]
      .apply(lambda x: (x == "Yes").mean())
      .reset_index(name="readmission_rate")
      .pivot(index="diagnosis", columns="department", values="readmission_rate")
)

fig_readmit_heatmap = px.imshow(
    readmit_heatmap_data,
    title="Readmission Rate — Diagnosis × Department",
    labels=dict(x="Department", y="Diagnosis", color="Readmission Rate"),
    color_continuous_scale="Reds",
    aspect="auto",
    text_auto=".0%"
)

fig_readmit_heatmap.update_layout(xaxis_tickangle=-45)

fig_readmit_heatmap.show()

## 3.21 Reamission Rate by Age Group and Smoker Status

In [82]:
readmit_by_age = (
    df.groupby("age_group", observed=True)["readmitted_30d"]
      .apply(lambda x: (x == "Yes").mean())
      .reset_index(name="readmission_rate")
)

fig_readmit_age = px.bar(
    readmit_by_age, x="age_group", y="readmission_rate",
    title="Readmission Rate by Age Group",
    color_discrete_sequence=[ALERT_COLOR], text="readmission_rate"
)
fig_readmit_age.update_traces(texttemplate="%{text:.1%}", textposition="outside")
fig_readmit_age.add_hline(y=readmission_benchmark, line_dash="dash", line_color="gray")
fig_readmit_age.update_layout(xaxis_title=None, yaxis_title="Readmission Rate", yaxis_tickformat=".0%")
fig_readmit_age.show()

readmit_by_smoker = (
    df.groupby("smoker")["readmitted_30d"]
      .apply(lambda x: (x == "Yes").mean())
      .reset_index(name="readmission_rate")
)

fig_readmit_smoker = px.bar(
    readmit_by_smoker, x="smoker", y="readmission_rate",
    title="Readmission Rate by Smoking Status",
    color_discrete_sequence=[ALERT_COLOR], text="readmission_rate"
)
fig_readmit_smoker.update_traces(texttemplate="%{text:.1%}", textposition="outside")
fig_readmit_smoker.add_hline(y=readmission_benchmark, line_dash="dash", line_color="gray")
fig_readmit_smoker.update_layout(xaxis_title=None, yaxis_title="Readmission Rate", yaxis_tickformat=".0%")
fig_readmit_smoker.show()

## 3.22 Length of Stay by Readmission Status

In [83]:
fig_los_readmit = px.box(
    df, x="readmitted_30d", y="length_of_stay",
    title="Length of Stay by Readmission Status",
    color="readmitted_30d",
    color_discrete_map={"No": PRIMARY_COLOR, "Yes": ALERT_COLOR},
    points=False
)

fig_los_readmit.update_layout(showlegend=False, xaxis_title=None, yaxis_title="Length of Stay (days)")

fig_los_readmit.show()

## 3.23 Readmission Rate of Emergency vs. Other Departments

In [84]:
df["is_emergency"] = np.where(df["department"] == "Emergency", "Emergency", "All Other Departments")

emergency_comparison = (
    df.groupby("is_emergency")
      .agg(
          avg_los=("length_of_stay", "mean"),
          avg_charges=("total_charges", "mean"),
          readmission_rate=("readmitted_30d", lambda x: (x == "Yes").mean())
      )
      .round(2)
      .reset_index()
)

emergency_comparison

,is_emergency,avg_los,avg_charges,readmission_rate
0,All Other Departments,6.89,"9,517,575.41",0.19
1,Emergency,6.38,"16,106,628.83",0.20


In [85]:
fig_emergency = px.bar(
    emergency_comparison, x="is_emergency", y="readmission_rate",
    title="Readmission Rate: Emergency vs. All Other Departments",
    color="is_emergency",
    color_discrete_map={"Emergency": ALERT_COLOR, "All Other Departments": PRIMARY_COLOR},
    text="readmission_rate"
)
fig_emergency.update_traces(texttemplate="%{text:.1%}", textposition="outside")
fig_emergency.add_hline(y=readmission_benchmark, line_dash="dash", line_color="gray")
fig_emergency.update_layout(showlegend=False, xaxis_title=None, yaxis_title="Readmission Rate", yaxis_tickformat=".0%")
fig_emergency.show()

## 3.24 Financial Impact

In [86]:
readmitted_revenue = df.loc[df["readmitted_30d"] == "Yes", "total_charges"].sum()
readmitted_count = (df["readmitted_30d"] == "Yes").sum()

excess_readmissions = max(0, readmitted_count - int(total_admissions * readmission_benchmark))
avg_readmit_charge = df.loc[df["readmitted_30d"] == "Yes", "total_charges"].mean()
estimated_excess_cost = excess_readmissions * avg_readmit_charge

print("=" * 55)
print("READMISSION — FINANCIAL IMPACT")
print("=" * 55)
print(f"Total charges tied to readmitted patients: IDR {readmitted_revenue:,.0f}")
print(f"Readmitted patient count:                  {readmitted_count:,}")
print(f"Excess readmissions vs. 15% benchmark:      {excess_readmissions:,}")
print(f"Estimated excess cost vs. benchmark:        IDR {estimated_excess_cost:,.0f}")

READMISSION — FINANCIAL IMPACT
Total charges tied to readmitted patients: IDR 19,673,572,802
Readmitted patient count:                  2,004
Excess readmissions vs. 15% benchmark:      429
Estimated excess cost vs. benchmark:        IDR 4,211,558,250


## 3.25 Age Distribution

In [87]:
fig_age_dist = px.histogram(
    df, x="age", color="age_group",
    title="Patient Age Distribution",
    nbins=40,
    color_discrete_sequence=px.colors.sequential.Teal
)

fig_age_dist.update_layout(
    xaxis_title="Age", yaxis_title="Number of Patients",
    legend_title="Age Group", bargap=0.05
)

fig_age_dist.show()

## 3.26 BMI Distribution by Department

In [88]:
fig_bmi_dept = px.box(
    df, x="department", y="bmi",
    title="BMI Distribution by Department",
    color="department",
    color_discrete_map=DEPARTMENT_COLORS,
    points=False
)

fig_bmi_dept.update_layout(showlegend=False, xaxis_title=None, yaxis_title="BMI")

fig_bmi_dept.show()

## 3.27 Smoker Percentage by Department

In [89]:
smoker_by_dept = (
    df.groupby("department")["smoker"]
      .apply(lambda x: (x == "Yes").mean())
      .reset_index(name="smoker_pct")
      .sort_values("smoker_pct", ascending=True)
)

fig_smoker_dept = px.bar(
    smoker_by_dept, x="smoker_pct", y="department", orientation="h",
    title="Smoker Percentage by Department",
    color_discrete_sequence=[PRIMARY_COLOR], text="smoker_pct"
)
fig_smoker_dept.update_traces(texttemplate="%{text:.1%}", textposition="outside")
fig_smoker_dept.update_layout(xaxis_title="% Smokers", yaxis_title=None, xaxis_tickformat=".0%")

fig_smoker_dept.show()

## 3.28 Gender Split

In [90]:
gender_counts = df["gender"].value_counts().reset_index()
gender_counts.columns = ["gender", "count"]

fig_gender = px.pie(
    gender_counts, values="count", names="gender",
    title="Patient Gender Split",
    color="gender",
    color_discrete_map={"Male": PRIMARY_COLOR, "Female": "#5a9bb0"},
    hole=0.5
)
fig_gender.update_traces(textinfo="label+percent")
fig_gender.update_layout(showlegend=False)

fig_gender.show()

## 3.29 Blood Type Distribution

In [91]:
blood_counts = df["blood_type"].value_counts().reset_index()
blood_counts.columns = ["blood_type", "count"]

fig_blood = px.bar(
    blood_counts.sort_values("count", ascending=True),
    x="count", y="blood_type", orientation="h",
    title="Blood Type Distribution",
    color_discrete_sequence=["#a3c9d1"]  # deliberately muted — supporting chart, not a headline
)
fig_blood.update_layout(xaxis_title="Patients", yaxis_title=None)

fig_blood.show()